# SafeTraq: Crash Detection Data Pipeline and Model Comparison

This notebook develops and evaluates a machine-learning pipeline for vehicle crash detection using multiple crash and normal-driving datasets and real-road sensor sessions.

## Pipeline overview

- Load and standardize sensor datasets.
- Merge sources while preserving session and source identifiers.
- Segment sessions when timestamp gaps exceed 0.3 seconds.
- Create 60-timestep sensor windows with 50% overlap using seven features: accelerometer XYZ, gyroscope XYZ, and speed.
- Split windows by group using `GroupShuffleSplit` to reduce session leakage.
- Prepare features and compare eight machine-learning and deep-learning approaches.
- Evaluate model predictions, including testing against real-road sensor sessions.

## Data sources

The notebook references datasets mounted in the Kaggle environment under `/kaggle/input/datasets/youssefashraf57/`, including crash-data collections, normal and highway driving data, and real-road sensor sessions.

**Reproducibility note:** These paths refer to Kaggle datasets and are not automatically available on a local computer. Confirm the exact source dataset versions, licenses, preprocessing steps, and saved outputs before claiming full reproducibility.

## Scope and limitations

This notebook documents the experimental pipeline. Results depend on the source data, preprocessing, model configuration, and evaluation procedure. Real-road testing is not equivalent to a fully independent, representative safety validation.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installedEJ;,VZ7.
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current sessio n

/kaggle/input/datasets/youssefashraf57/crash-data1/all_crash_data1.csv
/kaggle/input/datasets/youssefashraf57/crash-data1/normal_driving_data.csv
/kaggle/input/datasets/youssefashraf57/crash-data1/highway_driving_data.csv
/kaggle/input/datasets/youssefashraf57/crash-data1/traffic_jam_data.csv
/kaggle/input/datasets/youssefashraf57/crash-data/crash_data_log.csv
/kaggle/input/datasets/youssefashraf57/crash-data/all_crash_data.csv
/kaggle/input/datasets/youssefashraf57/crash-data/crash_data.csv
/kaggle/input/datasets/youssefashraf57/real-data/Road Data/Road Anomalies/2. Bump/Location.csv
/kaggle/input/datasets/youssefashraf57/real-data/Road Data/Road Anomalies/2. Bump/Accelerometer.csv
/kaggle/input/datasets/youssefashraf57/real-data/Road Data/Road Anomalies/2. Bump/GyroscopeUncalibrated.csv
/kaggle/input/datasets/youssefashraf57/real-data/Road Data/Road Anomalies/2. Bump/TotalAcceleration.csv
/kaggle/input/datasets/youssefashraf57/real-data/Road Data/Road Anomalies/2. Bump/AccelerometerU

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout, GRU
from tensorflow.keras.callbacks import EarlyStopping

2026-03-15 04:37:00.561945: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773549420.762966      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773549420.822202      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773549421.293315      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773549421.293360      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773549421.293367      23 computation_placer.cc:177] computation placer alr

In [ ]:
# =========================================================
# LOAD DATASETS
# =========================================================

# PC1
data_crash_main = pd.read_csv(
    "/kaggle/input/datasets/youssefashraf57/crash-data/crash_data.csv"
)

data_crash_log = pd.read_csv(
    "/kaggle/input/datasets/youssefashraf57/crash-data/crash_data_log.csv"
)

data_crash_combined = pd.read_csv(
    "/kaggle/input/datasets/youssefashraf57/crash-data/all_crash_data.csv"
)

# PC2
data_crash_extra = pd.read_csv(
    "/kaggle/input/datasets/youssefashraf57/crash-data1/all_crash_data1.csv",
    sep=";"
)

data_normal_driving = pd.read_csv(
    "/kaggle/input/datasets/youssefashraf57/crash-data1/normal_driving_data.csv",
    sep=";"
)

data_highway_driving = pd.read_csv(
    "/kaggle/input/datasets/youssefashraf57/crash-data1/highway_driving_data.csv",
    sep=";"
)

# traffic_jam_data.csv may be malformed as one single text column
data_traffic_jam = pd.read_csv(
    "/kaggle/input/datasets/youssefashraf57/crash-data1/traffic_jam_data.csv",
    sep=";"
)

In [ ]:
# =========================================================
# FIX TRAFFIC DATA IF IT WAS READ AS ONE TEXT COLUMN
# =========================================================

if data_traffic_jam.shape[1] == 1:
    raw_col = data_traffic_jam.columns[0]

    split_df = data_traffic_jam[raw_col].str.split(",", expand=True)

    split_df.columns = [
        "timestamp", "accel_x", "accel_y", "accel_z",
        "gyro_x", "gyro_y", "gyro_z", "speed", "collision"
    ]

    data_traffic_jam = split_df.copy()

print("traffic_jam shape after fix:", data_traffic_jam.shape)
print("traffic_jam columns:", data_traffic_jam.columns.tolist())

# Convert all columns safely
for col in data_traffic_jam.columns:
    data_traffic_jam[col] = pd.to_numeric(data_traffic_jam[col], errors="coerce")

print("traffic_jam shape after fix:", data_traffic_jam.shape)
print("traffic_jam columns:", data_traffic_jam.columns.tolist())
print("traffic_jam original collision distribution:")
print(data_traffic_jam["collision"].value_counts(dropna=False))

# IMPORTANT:
# This file is known to represent normal driving only.
# Therefore any positive collision labels are treated as annotation errors.
traffic_positive_count = int((data_traffic_jam["collision"] > 0).sum())

if traffic_positive_count > 0:
    print(f"WARNING: traffic_jam_data contains {traffic_positive_count} positive labels.")
    print("They will be forced to 0 because this dataset is supposed to be normal only.")

data_traffic_jam["collision"] = 0

traffic_jam shape after fix: (39476, 9)
traffic_jam columns: ['timestamp', 'accel_x', 'accel_y', 'accel_z', 'gyro_x', 'gyro_y', 'gyro_z', 'speed', 'collision']
traffic_jam shape after fix: (39476, 9)
traffic_jam columns: ['timestamp', 'accel_x', 'accel_y', 'accel_z', 'gyro_x', 'gyro_y', 'gyro_z', 'speed', 'collision']
traffic_jam original collision distribution:
collision
0    39461
1       15
Name: count, dtype: int64
They will be forced to 0 because this dataset is supposed to be normal only.


In [ ]:
# =========================================================
# PUT ALL DATASETS IN ONE DICTIONARY
# =========================================================

all_datasets = {
    "data_crash_main": data_crash_main,
    "data_crash_log": data_crash_log,
    "data_crash_combined": data_crash_combined,
    "data_crash_extra": data_crash_extra,
    "data_normal_driving": data_normal_driving,
    "data_highway_driving": data_highway_driving,
    "data_traffic_jam": data_traffic_jam
}

In [ ]:
# =========================================================
# INSPECT ORIGINAL DATASETS
# =========================================================

for name, df in all_datasets.items():
    print("\n" + "=" * 70)
    print(name)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(2))


data_crash_main
Shape: (100479, 9)
Columns: ['timestamp', 'accel_x', 'accel_y', 'accel_z', 'gyro_x', 'gyro_y', 'gyro_z', 'speed_ms', 'collision']
                    timestamp       accel_x       accel_y      accel_z  \
0  2026-02-26T17:11:10.317543 -3.346121e-36  6.176881e-36     9.810000   
1  2026-02-26T17:11:10.592832  1.138626e+04 -2.101884e+04 -1039.984131   

   gyro_x  gyro_y  gyro_z  speed_ms  collision  
0     0.0     0.0     0.0    0.4899          0  
1     0.0     0.0     0.0    0.9795          0  

data_crash_log
Shape: (20183, 12)
Columns: ['timestamp_iso', 'accel_x', 'accel_y', 'accel_z', 'gyro_x', 'gyro_y', 'gyro_z', 'lat', 'lon', 'alt', 'Speed', 'label']
                timestamp_iso   accel_x   accel_y   accel_z    gyro_x  \
0  2026-02-12T18:01:55.981649 -0.419305  0.443209 -0.000946 -0.023982   
1  2026-02-12T18:01:55.981649 -0.419305  0.443209 -0.000946 -0.023982   

     gyro_y    gyro_z       lat      lon       alt     Speed  label  
0  0.000274 -0.000114  0.0005

In [ ]:
# =========================================================
# STANDARDIZE ALL DATASETS
# Final columns:
# timestamp, accel_x, accel_y, accel_z,
# gyro_x, gyro_y, gyro_z, speed, collision
# =========================================================

final_columns = [
    "timestamp",
    "accel_x", "accel_y", "accel_z",
    "gyro_x", "gyro_y", "gyro_z",
    "speed",
    "collision"
]

feature_columns = [
    "accel_x", "accel_y", "accel_z",
    "gyro_x", "gyro_y", "gyro_z",
    "speed"
]

def convert_timestamp_to_seconds(series: pd.Series, dataset_name: str = "") -> pd.Series:
    s = series.copy()

    numeric = pd.to_numeric(s, errors="coerce")

    if numeric.notna().mean() >= 0.95:
        numeric = numeric.astype(float)
        first_valid = numeric.dropna().iloc[0]
        numeric = numeric - first_valid
        return numeric

    dt = pd.to_datetime(s, errors="coerce", utc=True)

    if dt.notna().sum() > 0:
        first_valid = dt.dropna().iloc[0]
        dt_seconds = (dt - first_valid).dt.total_seconds()
        return dt_seconds

    return pd.Series(np.nan, index=series.index)

def standardize_one_dataset(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    df = df.copy()

    # Rename possible column variants
    rename_map = {}

    if "speed_ms" in df.columns:
        rename_map["speed_ms"] = "speed"
    if "Speed" in df.columns:
        rename_map["Speed"] = "speed"
    if "label" in df.columns and "collision" not in df.columns:
        rename_map["label"] = "collision"
    if "timestamp_iso" in df.columns and "timestamp" not in df.columns:
        rename_map["timestamp_iso"] = "timestamp"

    df = df.rename(columns=rename_map)

    # Add any missing final columns
    for col in final_columns:
        if col not in df.columns:
            df[col] = np.nan

    # Keep only needed columns
    df = df[final_columns].copy()

    # Convert timestamp safely
    df["timestamp"] = convert_timestamp_to_seconds(df["timestamp"], dataset_name)

    # Convert numeric columns
    for col in feature_columns + ["collision"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Force collision to binary
    df["collision"] = df["collision"].fillna(0)
    df["collision"] = (df["collision"] > 0).astype(int)

    print(f"\n{dataset_name}")
    print("Shape after standardization:", df.shape)
    print("Missing timestamps:", df["timestamp"].isna().sum())
    print("Collision distribution:")
    print(df["collision"].value_counts(dropna=False))

    return df

standardized_datasets = {}

for dataset_name, df in all_datasets.items():
    standardized_datasets[dataset_name] = standardize_one_dataset(df, dataset_name)


data_crash_main
Shape after standardization: (100479, 9)
Missing timestamps: 0
Collision distribution:
collision
0    52143
1    48336
Name: count, dtype: int64

data_crash_log
Shape after standardization: (20183, 9)
Missing timestamps: 0
Collision distribution:
collision
0    18361
1     1822
Name: count, dtype: int64

data_crash_combined
Shape after standardization: (36222, 9)
Missing timestamps: 0
Collision distribution:
collision
1    21401
0    14821
Name: count, dtype: int64

data_crash_extra
Shape after standardization: (8602, 9)
Missing timestamps: 1929
Collision distribution:
collision
1    4823
0    3779
Name: count, dtype: int64

data_normal_driving
Shape after standardization: (12001, 9)
Missing timestamps: 0
Collision distribution:
collision
0    12001
Name: count, dtype: int64

data_highway_driving
Shape after standardization: (12001, 9)
Missing timestamps: 0
Collision distribution:
collision
0    12001
Name: count, dtype: int64

data_traffic_jam
Shape after standardizat

In [ ]:
# =========================================================
# CLEAN DATASETS
# =========================================================

def clean_sensor_data(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    df = df.copy()

    before = len(df)

    # Remove duplicates
    df = df.drop_duplicates()

    # Drop rows missing required data
    required_columns = final_columns
    df = df.dropna(subset=required_columns)

    # Sort by timestamp inside file
    df = df.sort_values("timestamp").reset_index(drop=True)

    after = len(df)

    print(f"\n{dataset_name}")
    print("Rows before:", before)
    print("Rows after :", after)
    print("Removed    :", before - after)

    return df

cleaned_datasets = {}

for dataset_name, df in standardized_datasets.items():
    cleaned_datasets[dataset_name] = clean_sensor_data(df, dataset_name)


data_crash_main
Rows before: 100479
Rows after : 100479
Removed    : 0

data_crash_log
Rows before: 20183
Rows after : 8119
Removed    : 12064

data_crash_combined
Rows before: 36222
Rows after : 36222
Removed    : 0

data_crash_extra
Rows before: 8602
Rows after : 6673
Removed    : 1929

data_normal_driving
Rows before: 12001
Rows after : 12001
Removed    : 0

data_highway_driving
Rows before: 12001
Rows after : 12001
Removed    : 0

data_traffic_jam
Rows before: 39476
Rows after : 39476
Removed    : 0


In [ ]:
# =========================================================
# ADD UNIQUE SESSION ID TO EACH ORIGINAL FILE
# =========================================================

for dataset_name in cleaned_datasets:
    cleaned_datasets[dataset_name]["session_id"] = dataset_name

# Optional source_id just for inspection
pc1_names = ["data_crash_main", "data_crash_log", "data_crash_combined"]
pc2_names = ["data_crash_extra", "data_normal_driving", "data_highway_driving", "data_traffic_jam"]

for dataset_name in cleaned_datasets:
    if dataset_name in pc1_names:
        cleaned_datasets[dataset_name]["source_id"] = "PC1"
    else:
        cleaned_datasets[dataset_name]["source_id"] = "PC2"

In [ ]:
# =========================================================
# MERGE INTO MASTER DATASET
# =========================================================

master_dataset = pd.concat(
    [cleaned_datasets[name] for name in cleaned_datasets],
    ignore_index=True
)

print("Master dataset shape:", master_dataset.shape)
print("\nCollision distribution:")
print(master_dataset["collision"].value_counts())

print("\nSession distribution:")
print(master_dataset["session_id"].value_counts())

print("\nSource distribution:")
print(master_dataset["source_id"].value_counts())

Master dataset shape: (214971, 11)

Collision distribution:
collision
0    139451
1     75520
Name: count, dtype: int64

Session distribution:
session_id
data_crash_main         100479
data_traffic_jam         39476
data_crash_combined      36222
data_highway_driving     12001
data_normal_driving      12001
data_crash_log            8119
data_crash_extra          6673
Name: count, dtype: int64

Source distribution:
source_id
PC1    144820
PC2     70151
Name: count, dtype: int64


In [ ]:
# =========================================================
# SEGMENT DATA USING TIME GAPS PER SESSION
# =========================================================

master_dataset = master_dataset.sort_values(
    ["session_id", "timestamp"]
).reset_index(drop=True)

master_dataset["dt"] = (
    master_dataset.groupby("session_id")["timestamp"]
    .diff()
    .fillna(0)
)

print(master_dataset["dt"].describe())

count    214971.000000
mean          0.089278
std           4.697387
min           0.000000
25%           0.041211
50%           0.050000
75%           0.054407
max        1380.464040
Name: dt, dtype: float64


In [ ]:
gap_threshold = 0.3  # adjust if needed

master_dataset["new_segment"] = master_dataset["dt"] > gap_threshold

master_dataset["segment_id"] = (
    master_dataset.groupby("session_id")["new_segment"]
    .cumsum()
)

master_dataset["segment_key"] = (
    master_dataset["session_id"].astype(str) + "_seg_" +
    master_dataset["segment_id"].astype(str)
)

print("Number of unique segment keys:", master_dataset["segment_key"].nunique())
print(master_dataset[["session_id", "segment_id", "segment_key"]].head())

Number of unique segment keys: 333
            session_id  segment_id                segment_key
0  data_crash_combined           0  data_crash_combined_seg_0
1  data_crash_combined           0  data_crash_combined_seg_0
2  data_crash_combined           0  data_crash_combined_seg_0
3  data_crash_combined           0  data_crash_combined_seg_0
4  data_crash_combined           0  data_crash_combined_seg_0


In [ ]:
# =========================================================
# OPTIONAL: LIGHT CLIPPING USING GLOBAL QUANTILES
# Better than hardcoded values, but still optional
# =========================================================

for col in ["accel_x", "accel_y", "accel_z", "gyro_x", "gyro_y", "gyro_z"]:
    low = master_dataset[col].quantile(0.001)
    high = master_dataset[col].quantile(0.999)
    master_dataset[col] = master_dataset[col].clip(low, high)

print("Optional clipping applied.")

Optional clipping applied.


In [ ]:
# =========================================================
# CREATE WINDOWS
# =========================================================

window_size = 60   # 3 seconds if effective frequency ~20 Hz
step_size = 30     # 50% overlap

X = []
y = []
window_groups = []

for segment_key in master_dataset["segment_key"].unique():
    segment_data = master_dataset[master_dataset["segment_key"] == segment_key].copy()

    if len(segment_data) < window_size:
        continue

    for start_idx in range(0, len(segment_data) - window_size + 1, step_size):
        window = segment_data.iloc[start_idx:start_idx + window_size]

        X.append(window[feature_columns].values)
        y.append(window["collision"].max())   # crash if any row inside window is crash
        window_groups.append(segment_key)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)
window_groups = np.array(window_groups)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Groups shape:", window_groups.shape)
print("Window label distribution:", np.unique(y, return_counts=True))

X shape: (6830, 60, 7)
y shape: (6830,)
Groups shape: (6830,)
Window label distribution: (array([0, 1], dtype=int32), array([3383, 3447]))


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

chosen_seed = 68

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=chosen_seed)
train_idx, test_idx = next(gss.split(X, y, groups=window_groups))

X_train = X[train_idx]
X_test = X[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

groups_train = window_groups[train_idx]
groups_test = window_groups[test_idx]

print("Chosen seed:", chosen_seed)
print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

print("Train label distribution:", np.unique(y_train, return_counts=True))
print("Test  label distribution:", np.unique(y_test, return_counts=True))

print("Unique train groups:", len(np.unique(groups_train)))
print("Unique test groups :", len(np.unique(groups_test)))

print("Common groups between train and test:",
      len(set(groups_train).intersection(set(groups_test))))

Chosen seed: 68
X_train shape: (5402, 60, 7)
X_test shape : (1428, 60, 7)
Train label distribution: (array([0, 1], dtype=int32), array([2674, 2728]))
Test  label distribution: (array([0, 1], dtype=int32), array([709, 719]))
Unique train groups: 160
Unique test groups : 41
Common groups between train and test: 0


In [ ]:
# =========================================================
# SCALE USING TRAINING DATA ONLY
# =========================================================

scaler = StandardScaler()

# Flatten to 2D for fitting scaler
X_train_2d = X_train.reshape(-1, len(feature_columns))
X_test_2d = X_test.reshape(-1, len(feature_columns))

scaler.fit(X_train_2d)

X_train_scaled = scaler.transform(X_train_2d).reshape(X_train.shape)
X_test_scaled = scaler.transform(X_test_2d).reshape(X_test.shape)

print("Scaling completed without leakage.")

Scaling completed without leakage.


In [ ]:
# =========================================================
# EVALUATION HELPER
# =========================================================

def evaluate_predictions(y_true, y_pred, y_prob=None, model_name="Model"):
    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

    if y_prob is not None:
        try:
            roc = roc_auc_score(y_true, y_prob)
            pr_auc = average_precision_score(y_true, y_prob)
            print(f"ROC-AUC  : {roc:.4f}")
            print(f"PR-AUC   : {pr_auc:.4f}")
        except:
            pass

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

In [ ]:
# =========================================================
# FLATTEN WINDOWS FOR CLASSICAL ML MODELS
# =========================================================

X_train_flat = X_train_scaled.reshape(X_train_scaled.shape[0], -1)
X_test_flat = X_test_scaled.reshape(X_test_scaled.shape[0], -1)

print("X_train_flat:", X_train_flat.shape)
print("X_test_flat :", X_test_flat.shape)

X_train_flat: (5402, 420)
X_test_flat : (1428, 420)


In [ ]:
# =========================================================
# KNN
# =========================================================

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_flat, y_train)

y_pred_knn = knn_model.predict(X_test_flat)

# KNN probability
if hasattr(knn_model, "predict_proba"):
    y_prob_knn = knn_model.predict_proba(X_test_flat)[:, 1]
else:
    y_prob_knn = None

evaluate_predictions(y_test, y_pred_knn, y_prob_knn, model_name="KNN")


KNN
Accuracy : 0.7087
Precision: 0.7663
Recall   : 0.6064
F1-score : 0.6770
ROC-AUC  : 0.7981
PR-AUC   : 0.7598

Confusion Matrix:
[[576 133]
 [283 436]]

Classification Report:
              precision    recall  f1-score   support

           0     0.6705    0.8124    0.7347       709
           1     0.7663    0.6064    0.6770       719

    accuracy                         0.7087      1428
   macro avg     0.7184    0.7094    0.7059      1428
weighted avg     0.7187    0.7087    0.7057      1428



In [ ]:
# =========================================================
# SVM
# =========================================================

svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    random_state=42
)

svm_model.fit(X_train_flat, y_train)

y_pred_svm = svm_model.predict(X_test_flat)
y_prob_svm = svm_model.predict_proba(X_test_flat)[:, 1]

evaluate_predictions(y_test, y_pred_svm, y_prob_svm, model_name="SVM")


SVM
Accuracy : 0.7437
Precision: 0.7421
Recall   : 0.7524
F1-score : 0.7472
ROC-AUC  : 0.8594
PR-AUC   : 0.8533

Confusion Matrix:
[[521 188]
 [178 541]]

Classification Report:
              precision    recall  f1-score   support

           0     0.7454    0.7348    0.7401       709
           1     0.7421    0.7524    0.7472       719

    accuracy                         0.7437      1428
   macro avg     0.7437    0.7436    0.7436      1428
weighted avg     0.7437    0.7437    0.7437      1428



In [ ]:
# =========================================================
# RANDOM FOREST
# =========================================================

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_model.fit(X_train_flat, y_train)

y_pred_rf = rf_model.predict(X_test_flat)
y_prob_rf = rf_model.predict_proba(X_test_flat)[:, 1]

evaluate_predictions(y_test, y_pred_rf, y_prob_rf, model_name="Random Forest")


Random Forest
Accuracy : 0.8053
Precision: 0.7537
Recall   : 0.9110
F1-score : 0.8249
ROC-AUC  : 0.9154
PR-AUC   : 0.9103

Confusion Matrix:
[[495 214]
 [ 64 655]]

Classification Report:
              precision    recall  f1-score   support

           0     0.8855    0.6982    0.7808       709
           1     0.7537    0.9110    0.8249       719

    accuracy                         0.8053      1428
   macro avg     0.8196    0.8046    0.8028      1428
weighted avg     0.8192    0.8053    0.8030      1428



In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

lstm_model = Sequential()

lstm_model.add(
    LSTM(
        64,
        return_sequences=True,
        input_shape=(X_train_scaled.shape[1], X_train_scaled.shape[2])
    )
)

lstm_model.add(Dropout(0.3))

lstm_model.add(LSTM(32))

lstm_model.add(Dropout(0.3))

lstm_model.add(Dense(16, activation="relu"))
lstm_model.add(Dense(1, activation="sigmoid"))

lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_model.summary()

I0000 00:00:1773549484.989363      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773549484.995421      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 64)         │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,393 (122.63 KB)

 Trainable params: 31,393 (122.63 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history_lstm = lstm_model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50


I0000 00:00:1773549489.080645      90 cuda_dnn.cc:529] Loaded cuDNN version 91002


136/136 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.7010 - loss: 0.6083 - val_accuracy: 0.9528 - val_loss: 0.5036
Epoch 2/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7437 - loss: 0.5015 - val_accuracy: 0.9066 - val_loss: 0.5347
Epoch 3/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7735 - loss: 0.4590 - val_accuracy: 0.9907 - val_loss: 0.2792
Epoch 4/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7514 - loss: 0.4713 - val_accuracy: 0.9140 - val_loss: 0.3985
Epoch 5/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7782 - loss: 0.4398 - val_accuracy: 0.9833 - val_loss: 0.2986
Epoch 6/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7814 - loss: 0.4272 - val_accuracy: 0.9621 - val_loss: 0.2790
Epoch 7/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7994 - loss: 0.4019 - val_accuracy: 0.9741 - val_loss: 0.2401
Epoch 8/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8109 - loss: 0.3867 - val_accuracy: 0.977

In [ ]:
y_prob_lstm = lstm_model.predict(X_test_scaled).ravel()
y_pred_lstm = (y_prob_lstm >= 0.5).astype(int)

evaluate_predictions(y_test, y_pred_lstm, y_prob_lstm, "LSTM")

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step

LSTM
Accuracy : 0.7780
Precision: 0.7956
Recall   : 0.7524
F1-score : 0.7734
ROC-AUC  : 0.8928
PR-AUC   : 0.8950

Confusion Matrix:
[[570 139]
 [178 541]]

Classification Report:
              precision    recall  f1-score   support

           0     0.7620    0.8039    0.7824       709
           1     0.7956    0.7524    0.7734       719

    accuracy                         0.7780      1428
   macro avg     0.7788    0.7782    0.7779      1428
weighted avg     0.7789    0.7780    0.7779      1428



In [ ]:
# =========================================================
# CNN
# =========================================================

tf.keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)

cnn_model = Sequential([
    Input(shape=(window_size, len(feature_columns))),

    Conv1D(filters=64, kernel_size=3, activation="relu"),
    MaxPooling1D(pool_size=2),

    Conv1D(filters=128, kernel_size=3, activation="relu"),
    MaxPooling1D(pool_size=2),

    Flatten(),

    Dense(128, activation="relu"),
    Dropout(0.3),

    Dense(1, activation="sigmoid")
])

cnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

cnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 58, 64)         │         1,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 29, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 27, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 13, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1664)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       213,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 239,361 (935.00 KB)

 Trainable params: 239,361 (935.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history_cnn = cnn_model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/30


I0000 00:00:1773549516.342664      91 service.cc:152] XLA service 0x7dd36c348460 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773549516.342703      91 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1773549516.342707      91 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5


 65/136 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7079 - loss: 0.5261

I0000 00:00:1773549518.940466      91 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


136/136 ━━━━━━━━━━━━━━━━━━━━ 7s 20ms/step - accuracy: 0.7367 - loss: 0.5039 - val_accuracy: 0.9750 - val_loss: 0.4052
Epoch 2/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7921 - loss: 0.4193 - val_accuracy: 0.9704 - val_loss: 0.2999
Epoch 3/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7996 - loss: 0.3894 - val_accuracy: 0.9907 - val_loss: 0.2159
Epoch 4/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8047 - loss: 0.3624 - val_accuracy: 0.9759 - val_loss: 0.1975
Epoch 5/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8185 - loss: 0.3339 - val_accuracy: 0.9861 - val_loss: 0.1579
Epoch 6/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8242 - loss: 0.3188 - val_accuracy: 0.9861 - val_loss: 0.1471
Epoch 7/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8403 - loss: 0.3067 - val_accuracy: 1.0000 - val_loss: 0.1220
Epoch 8/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8459 - loss: 0.2875 - val_accuracy: 0.9991 - val

In [ ]:
y_prob_cnn = cnn_model.predict(X_test_scaled).ravel()
y_pred_cnn = (y_prob_cnn >= 0.5).astype(int)

evaluate_predictions(y_test, y_pred_cnn, y_prob_cnn, model_name="CNN")

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step

CNN
Accuracy : 0.7927
Precision: 0.8180
Recall   : 0.7566
F1-score : 0.7861
ROC-AUC  : 0.8997
PR-AUC   : 0.8931

Confusion Matrix:
[[588 121]
 [175 544]]

Classification Report:
              precision    recall  f1-score   support

           0     0.7706    0.8293    0.7989       709
           1     0.8180    0.7566    0.7861       719

    accuracy                         0.7927      1428
   macro avg     0.7943    0.7930    0.7925      1428
weighted avg     0.7945    0.7927    0.7925      1428



In [ ]:
# =========================================================
# CNN + SVM
# =========================================================

cnn_feature_extractor = Model(
    inputs=cnn_model.inputs,
    outputs=cnn_model.layers[-3].output   # Dense(128)
)

X_train_cnn_features = cnn_feature_extractor.predict(X_train_scaled)
X_test_cnn_features = cnn_feature_extractor.predict(X_test_scaled)

print("CNN feature train shape:", X_train_cnn_features.shape)
print("CNN feature test  shape:", X_test_cnn_features.shape)

169/169 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
CNN feature train shape: (5402, 128)
CNN feature test  shape: (1428, 128)


In [ ]:
cnn_svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    random_state=42
)

cnn_svm_model.fit(X_train_cnn_features, y_train)

y_pred_cnn_svm = cnn_svm_model.predict(X_test_cnn_features)
y_prob_cnn_svm = cnn_svm_model.predict_proba(X_test_cnn_features)[:, 1]

evaluate_predictions(y_test, y_pred_cnn_svm, y_prob_cnn_svm, model_name="CNN + SVM")


CNN + SVM
Accuracy : 0.7815
Precision: 0.7980
Recall   : 0.7580
F1-score : 0.7775
ROC-AUC  : 0.8916
PR-AUC   : 0.8846

Confusion Matrix:
[[571 138]
 [174 545]]

Classification Report:
              precision    recall  f1-score   support

           0     0.7664    0.8054    0.7854       709
           1     0.7980    0.7580    0.7775       719

    accuracy                         0.7815      1428
   macro avg     0.7822    0.7817    0.7814      1428
weighted avg     0.7823    0.7815    0.7814      1428



In [ ]:
# =========================================================
# CNN + GRU
# =========================================================

tf.keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)

cnn_gru_model = Sequential([
    Input(shape=(window_size, len(feature_columns))),

    Conv1D(filters=64, kernel_size=3, activation="relu"),
    MaxPooling1D(pool_size=2),

    Conv1D(filters=128, kernel_size=3, activation="relu"),
    MaxPooling1D(pool_size=2),

    GRU(64, return_sequences=False),

    Dense(64, activation="relu"),
    Dropout(0.3),

    Dense(1, activation="sigmoid")
])

cnn_gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

cnn_gru_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 58, 64)         │         1,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 29, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 27, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 13, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 67,585 (264.00 KB)

 Trainable params: 67,585 (264.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history_cnn_gru = cnn_gru_model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.7186 - loss: 0.5149 - val_accuracy: 0.9917 - val_loss: 0.2482
Epoch 2/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8008 - loss: 0.4086 - val_accuracy: 0.9981 - val_loss: 0.1781
Epoch 3/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8075 - loss: 0.3778 - val_accuracy: 1.0000 - val_loss: 0.1008
Epoch 4/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8038 - loss: 0.3555 - val_accuracy: 1.0000 - val_loss: 0.0876
Epoch 5/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8094 - loss: 0.3408 - val_accuracy: 1.0000 - val_loss: 0.0689
Epoch 6/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8125 - loss: 0.3296 - val_accuracy: 1.0000 - val_loss: 0.0604
Epoch 7/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8092 - loss: 0.3360 - val_accuracy: 1.0000 - val_loss: 0.0587
Epoch 8/30
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8209 - loss: 0.3193 - val_accuracy: 1

In [ ]:
y_prob_cnn_gru = cnn_gru_model.predict(X_test_scaled).ravel()
y_pred_cnn_gru = (y_prob_cnn_gru >= 0.5).astype(int)

evaluate_predictions(y_test, y_pred_cnn_gru, y_prob_cnn_gru, model_name="CNN + GRU")

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step

CNN + GRU
Accuracy : 0.7829
Precision: 0.7337
Recall   : 0.8929
F1-score : 0.8055
ROC-AUC  : 0.9033
PR-AUC   : 0.9065

Confusion Matrix:
[[476 233]
 [ 77 642]]

Classification Report:
              precision    recall  f1-score   support

           0     0.8608    0.6714    0.7544       709
           1     0.7337    0.8929    0.8055       719

    accuracy                         0.7829      1428
   macro avg     0.7972    0.7821    0.7799      1428
weighted avg     0.7968    0.7829    0.7801      1428



In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

cnn_lstm_model = Sequential()

# CNN feature extractor
cnn_lstm_model.add(
    Conv1D(
        filters=64,
        kernel_size=3,
        activation="relu",
        input_shape=(X_train_scaled.shape[1], X_train_scaled.shape[2])
    )
)

cnn_lstm_model.add(MaxPooling1D(pool_size=2))

cnn_lstm_model.add(Dropout(0.3))

# LSTM temporal learning
cnn_lstm_model.add(LSTM(64))

cnn_lstm_model.add(Dropout(0.3))

# Dense classifier
cnn_lstm_model.add(Dense(32, activation="relu"))
cnn_lstm_model.add(Dense(1, activation="sigmoid"))

cnn_lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

cnn_lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_2 (Conv1D)               │ (None, 58, 64)         │         1,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 29, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 29, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 36,545 (142.75 KB)

 Trainable params: 36,545 (142.75 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history_cnn_lstm = cnn_lstm_model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.6934 - loss: 0.5459 - val_accuracy: 0.9991 - val_loss: 0.2349
Epoch 2/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8021 - loss: 0.4109 - val_accuracy: 0.9991 - val_loss: 0.1256
Epoch 3/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8061 - loss: 0.3865 - val_accuracy: 1.0000 - val_loss: 0.0849
Epoch 4/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8129 - loss: 0.3717 - val_accuracy: 1.0000 - val_loss: 0.0997
Epoch 5/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8211 - loss: 0.3564 - val_accuracy: 1.0000 - val_loss: 0.0852
Epoch 6/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8188 - loss: 0.3482 - val_accuracy: 0.9926 - val_loss: 0.1079
Epoch 7/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8240 - loss: 0.3415 - val_accuracy: 0.9991 - val_loss: 0.1216
Epoch 8/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8130 - loss: 0.3354 - val_accuracy: 0

In [ ]:
y_prob_cnn_lstm = cnn_lstm_model.predict(X_test_scaled).ravel()
y_pred_cnn_lstm = (y_prob_cnn_lstm >= 0.5).astype(int)

evaluate_predictions(y_test, y_pred_cnn_lstm, y_prob_cnn_lstm, "CNN + LSTM")

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step

CNN + LSTM
Accuracy : 0.7815
Precision: 0.7674
Recall   : 0.8122
F1-score : 0.7892
ROC-AUC  : 0.8966
PR-AUC   : 0.8922

Confusion Matrix:
[[532 177]
 [135 584]]

Classification Report:
              precision    recall  f1-score   support

           0     0.7976    0.7504    0.7733       709
           1     0.7674    0.8122    0.7892       719

    accuracy                         0.7815      1428
   macro avg     0.7825    0.7813    0.7812      1428
weighted avg     0.7824    0.7815    0.7813      1428



In [ ]:
# =========================================================
# FINAL COMPARISON TABLE
# =========================================================

results = []

def add_result(name, y_true, y_pred, y_prob=None):
    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
    }

    if y_prob is not None:
        try:
            row["ROC-AUC"] = roc_auc_score(y_true, y_prob)
        except:
            row["ROC-AUC"] = np.nan

        try:
            row["PR-AUC"] = average_precision_score(y_true, y_prob)
        except:
            row["PR-AUC"] = np.nan
    else:
        row["ROC-AUC"] = np.nan
        row["PR-AUC"] = np.nan

    results.append(row)


add_result("KNN", y_test, y_pred_knn, y_prob_knn)
add_result("SVM", y_test, y_pred_svm, y_prob_svm)
add_result("Random Forest", y_test, y_pred_rf, y_prob_rf)
add_result("CNN", y_test, y_pred_cnn, y_prob_cnn)
add_result("CNN + SVM", y_test, y_pred_cnn_svm, y_prob_cnn_svm)
add_result("CNN + GRU", y_test, y_pred_cnn_gru, y_prob_cnn_gru)
add_result("LSTM", y_test, y_pred_lstm, y_prob_lstm)
add_result("CNN + LSTM", y_test, y_pred_cnn_lstm, y_prob_cnn_lstm)

results_df = pd.DataFrame(results)\
    .sort_values(by="F1", ascending=False)\
    .reset_index(drop=True)

results_df

,Model,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Random Forest,0.805322,0.753740,0.910987,0.824937,0.915387,0.910342
1,CNN + GRU,0.782913,0.733714,0.892907,0.805521,0.903274,0.906523
2,CNN + LSTM,0.781513,0.767411,0.812239,0.789189,0.896630,0.892241
3,CNN,0.792717,0.818045,0.756606,0.786127,0.899689,0.893138
4,CNN + SVM,0.781513,0.797950,0.757997,0.777461,0.891624,0.884605
5,LSTM,0.778011,0.795588,0.752434,0.773410,0.892842,0.894993
6,SVM,0.743697,0.742112,0.752434,0.747238,0.859372,0.853338
7,KNN,0.708683,0.766257,0.606398,0.677019,0.798074,0.759777


In [ ]:
# # =========================================================
# # SAVE MODEL + SCALER
# # =========================================================

# cnn_gru_model.save("/kaggle/working/cnn_gru_crash_model.h5")

# import joblib
# joblib.dump(scaler, "/kaggle/working/feature_scaler.pkl")

# print("Saved model and scaler.")

In [ ]:
# # =========================================================
# # CONVERT TO TFLITE
# # =========================================================

# converter = tf.lite.TFLiteConverter.from_keras_model(cnn_gru_model)
# tflite_model = converter.convert()

# with open("/kaggle/working/cnn_gru_crash_model.tflite", "wb") as f:
#     f.write(tflite_model)

# print("TFLite model saved.")

In [ ]:
# =========================================================
# TEST ALL REAL SESSIONS ON KAGGLE USING RANDOM FOREST
# =========================================================

import pandas as pd
import numpy as np
import os
from glob import glob

# ---------------------------------------------------------
# PATH
# ---------------------------------------------------------
base_path = "/kaggle/input/datasets/youssefashraf57/real-data/Road Data"

# ---------------------------------------------------------
# FINAL MODEL
# ---------------------------------------------------------
final_model = rf_model

# ---------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------
window_size = 60
step_size = 30

feature_columns = [
    "accel_x", "accel_y", "accel_z",
    "gyro_x", "gyro_y", "gyro_z",
    "speed"
]

# ---------------------------------------------------------
# FIND ALL VALID SESSION FOLDERS
# ---------------------------------------------------------
all_accel_files = glob(os.path.join(base_path, "**", "Accelerometer.csv"), recursive=True)

session_folders = []
for accel_file in all_accel_files:
    folder = os.path.dirname(accel_file)

    gyro_file = os.path.join(folder, "Gyroscope.csv")
    loc_file = os.path.join(folder, "Location.csv")

    if os.path.exists(gyro_file) and os.path.exists(loc_file):
        session_folders.append(folder)

session_folders = sorted(list(set(session_folders)))

print("Total sessions found:", len(session_folders))
for s in session_folders:
    print(s)

# ---------------------------------------------------------
# LOAD AND PREPARE ONE SESSION
# ---------------------------------------------------------
def load_and_prepare_session(folder):
    accel = pd.read_csv(os.path.join(folder, "Accelerometer.csv"))
    gyro  = pd.read_csv(os.path.join(folder, "Gyroscope.csv"))
    loc   = pd.read_csv(os.path.join(folder, "Location.csv"))

    # rename columns to match training data
    accel = accel.rename(columns={
        "seconds_elapsed": "timestamp",
        "x": "accel_x",
        "y": "accel_y",
        "z": "accel_z"
    })

    gyro = gyro.rename(columns={
        "seconds_elapsed": "timestamp",
        "x": "gyro_x",
        "y": "gyro_y",
        "z": "gyro_z"
    })

    loc = loc.rename(columns={
        "seconds_elapsed": "timestamp"
    })

    # keep only required columns
    accel = accel[["timestamp", "accel_x", "accel_y", "accel_z"]].copy()
    gyro  = gyro[["timestamp", "gyro_x", "gyro_y", "gyro_z"]].copy()
    loc   = loc[["timestamp", "speed"]].copy()

    # numeric conversion
    for col in accel.columns:
        accel[col] = pd.to_numeric(accel[col], errors="coerce")

    for col in gyro.columns:
        gyro[col] = pd.to_numeric(gyro[col], errors="coerce")

    for col in loc.columns:
        loc[col] = pd.to_numeric(loc[col], errors="coerce")

    # sort for merge_asof
    accel = accel.sort_values("timestamp").reset_index(drop=True)
    gyro  = gyro.sort_values("timestamp").reset_index(drop=True)
    loc   = loc.sort_values("timestamp").reset_index(drop=True)

    # merge by nearest timestamp
    merged = pd.merge_asof(accel, gyro, on="timestamp", direction="nearest")
    merged = pd.merge_asof(merged, loc, on="timestamp", direction="nearest")

    # drop missing rows
    merged = merged.dropna().reset_index(drop=True)

    return merged

# ---------------------------------------------------------
# CREATE WINDOWS
# ---------------------------------------------------------
def create_windows(df, feature_columns, window_size=60, step_size=30):
    X = []
    time_ranges = []

    if len(df) < window_size:
        return np.array([]), []

    for start_idx in range(0, len(df) - window_size + 1, step_size):
        window = df.iloc[start_idx:start_idx + window_size]

        X.append(window[feature_columns].values.astype(np.float32))
        time_ranges.append((
            float(window["timestamp"].iloc[0]),
            float(window["timestamp"].iloc[-1])
        ))

    return np.array(X, dtype=np.float32), time_ranges

# ---------------------------------------------------------
# SESSION TYPE HELPER
# ---------------------------------------------------------
def get_session_type(folder_path):
    path_lower = folder_path.lower()
    if "aggressive" in path_lower:
        return "Aggressive"
    elif "standard" in path_lower:
        return "Standard"
    elif "slow" in path_lower:
        return "Slow"
    elif "bump" in path_lower:
        return "Bump"
    elif "pothole" in path_lower:
        return "Pothole"
    else:
        return "Unknown"

# ---------------------------------------------------------
# RUN ALL SESSIONS
# ---------------------------------------------------------
all_session_results = []
all_window_results = []

for folder in session_folders:
    print(f"\nProcessing: {folder}")

    try:
        real_df = load_and_prepare_session(folder)

        if len(real_df) == 0:
            all_session_results.append({
                "session": folder,
                "session_type": get_session_type(folder),
                "num_rows": 0,
                "num_windows": 0,
                "predicted_crash_windows": 0,
                "crash_window_ratio": np.nan,
                "max_crash_probability": np.nan,
                "mean_crash_probability": np.nan,
                "status": "empty_after_merge"
            })
            continue

        X_real, window_time_ranges = create_windows(
            real_df,
            feature_columns=feature_columns,
            window_size=window_size,
            step_size=step_size
        )

        if len(X_real) == 0:
            all_session_results.append({
                "session": folder,
                "session_type": get_session_type(folder),
                "num_rows": len(real_df),
                "num_windows": 0,
                "predicted_crash_windows": 0,
                "crash_window_ratio": np.nan,
                "max_crash_probability": np.nan,
                "mean_crash_probability": np.nan,
                "status": "too_short_for_windowing"
            })
            continue

        # scale using the same scaler from training
        X_real_2d = X_real.reshape(-1, len(feature_columns))
        X_real_scaled = scaler.transform(X_real_2d).reshape(X_real.shape)

        # flatten for Random Forest
        X_real_flat = X_real_scaled.reshape(X_real_scaled.shape[0], -1)

        # predict
        y_pred_real = final_model.predict(X_real_flat)
        y_prob_real = final_model.predict_proba(X_real_flat)[:, 1]

        # per-window results
        temp_windows = pd.DataFrame({
            "session": folder,
            "session_type": get_session_type(folder),
            "window_index": np.arange(len(y_pred_real)),
            "start_time": [x[0] for x in window_time_ranges],
            "end_time": [x[1] for x in window_time_ranges],
            "crash_probability": y_prob_real,
            "predicted_label": y_pred_real
        })
        all_window_results.append(temp_windows)

        # per-session results
        all_session_results.append({
            "session": folder,
            "session_type": get_session_type(folder),
            "num_rows": len(real_df),
            "num_windows": len(y_pred_real),
            "predicted_crash_windows": int(np.sum(y_pred_real)),
            "crash_window_ratio": float(np.mean(y_pred_real)),
            "max_crash_probability": float(np.max(y_prob_real)),
            "mean_crash_probability": float(np.mean(y_prob_real)),
            "status": "ok"
        })

    except Exception as e:
        all_session_results.append({
            "session": folder,
            "session_type": get_session_type(folder),
            "num_rows": np.nan,
            "num_windows": np.nan,
            "predicted_crash_windows": np.nan,
            "crash_window_ratio": np.nan,
            "max_crash_probability": np.nan,
            "mean_crash_probability": np.nan,
            "status": f"error: {str(e)}"
        })

# ---------------------------------------------------------
# BUILD RESULTS TABLES
# ---------------------------------------------------------
session_results_df = pd.DataFrame(all_session_results)

if len(all_window_results) > 0:
    window_results_df = pd.concat(all_window_results, ignore_index=True)
else:
    window_results_df = pd.DataFrame()

# ---------------------------------------------------------
# DISPLAY RESULTS
# ---------------------------------------------------------
print("\n================ SESSION SUMMARY ================\n")
display(
    session_results_df.sort_values(
        by=["predicted_crash_windows", "max_crash_probability"],
        ascending=[False, False]
    ).reset_index(drop=True)
)

print("\n================ WINDOW RESULTS SAMPLE ================\n")
display(window_results_df.head(20))

print("\n================ SUMMARY BY SESSION TYPE ================\n")
if len(session_results_df) > 0:
    summary_by_type = (
        session_results_df.groupby("session_type", dropna=False)
        .agg(
            sessions=("session", "count"),
            total_windows=("num_windows", "sum"),
            predicted_crash_windows=("predicted_crash_windows", "sum"),
            avg_crash_ratio=("crash_window_ratio", "mean"),
            avg_max_probability=("max_crash_probability", "mean"),
            avg_mean_probability=("mean_crash_probability", "mean")
        )
        .reset_index()
    )
    display(summary_by_type)

print("\n================ ONLY SESSIONS WITH PREDICTED CRASH WINDOWS ================\n")
display(
    session_results_df[
        session_results_df["predicted_crash_windows"].fillna(0) > 0
    ].sort_values(
        by=["predicted_crash_windows", "max_crash_probability"],
        ascending=[False, False]
    ).reset_index(drop=True)
)

# # ---------------------------------------------------------
# # SAVE RESULTS
# # ---------------------------------------------------------
# session_results_path = "/kaggle/working/all_real_sessions_summary.csv"
# window_results_path = "/kaggle/working/all_real_windows_predictions.csv"

# session_results_df.to_csv(session_results_path, index=False)
# window_results_df.to_csv(window_results_path, index=False)

# print("\nSaved files:")
# print(session_results_path)
# print(window_results_path)

Total sessions found: 23
/kaggle/input/datasets/youssefashraf57/real-data/Road Data/Driving Behaviour/1. Aggressive
/kaggle/input/datasets/youssefashraf57/real-data/Road Data/Driving Behaviour/2. Aggressive
/kaggle/input/datasets/youssefashraf57/real-data/Road Data/Driving Behaviour/3. Standard
/kaggle/input/datasets/youssefashraf57/real-data/Road Data/Driving Behaviour/4. Standard
/kaggle/input/datasets/youssefashraf57/real-data/Road Data/Driving Behaviour/5. Slow
/kaggle/input/datasets/youssefashraf57/real-data/Road Data/Driving Behaviour/6. Slow
/kaggle/input/datasets/youssefashraf57/real-data/Road Data/Road Anomalies/1. Bump
/kaggle/input/datasets/youssefashraf57/real-data/Road Data/Road Anomalies/10. Pothole
/kaggle/input/datasets/youssefashraf57/real-data/Road Data/Road Anomalies/11. Pothole
/kaggle/input/datasets/youssefashraf57/real-data/Road Data/Road Anomalies/12. Pothole
/kaggle/input/datasets/youssefashraf57/real-data/Road Data/Road Anomalies/13. Pothole
/kaggle/input/datas

,session,session_type,num_rows,num_windows,predicted_crash_windows,crash_window_ratio,max_crash_probability,mean_crash_probability,status
0,/kaggle/input/datasets/youssefashraf57/real-da...,Slow,52101.0,1735.0,1735.0,1.0,1.000,0.977928,ok
1,/kaggle/input/datasets/youssefashraf57/real-da...,Slow,49197.0,1638.0,1638.0,1.0,1.000,0.952953,ok
2,/kaggle/input/datasets/youssefashraf57/real-da...,Aggressive,37501.0,1249.0,1249.0,1.0,1.000,0.930440,ok
3,/kaggle/input/datasets/youssefashraf57/real-da...,Bump,36133.0,1203.0,1203.0,1.0,1.000,0.976101,ok
4,/kaggle/input/datasets/youssefashraf57/real-da...,Pothole,31901.0,1062.0,1062.0,1.0,1.000,0.948559,ok
5,/kaggle/input/datasets/youssefashraf57/real-da...,Standard,20601.0,685.0,685.0,1.0,1.000,0.923916,ok
6,/kaggle/input/datasets/youssefashraf57/real-da...,Standard,16001.0,532.0,532.0,1.0,1.000,0.926057,ok
7,/kaggle/input/datasets/youssefashraf57/real-da...,Bump,10001.0,332.0,332.0,1.0,1.000,0.963901,ok
8,/kaggle/input/datasets/youssefashraf57/real-da...,Pothole,6701.0,222.0,222.0,1.0,1.000,0.971644,ok
9,/kaggle/input/datasets/youssefashraf57/real-da...,Pothole,3601.0,119.0,119.0,1.0,0.995,0.972773,ok



================ WINDOW RESULTS SAMPLE ================



,session,session_type,window_index,start_time,end_time,crash_probability,predicted_label
0,/kaggle/input/datasets/youssefashraf57/real-da...,Aggressive,0,0.164552,0.755402,0.990,1
1,/kaggle/input/datasets/youssefashraf57/real-da...,Aggressive,1,0.465013,1.055805,0.995,1
2,/kaggle/input/datasets/youssefashraf57/real-da...,Aggressive,2,0.765415,1.356208,0.965,1
3,/kaggle/input/datasets/youssefashraf57/real-da...,Aggressive,3,1.065819,1.656612,0.955,1
4,/kaggle/input/datasets/youssefashraf57/real-da...,Aggressive,4,1.366222,1.957016,0.955,1
5,/kaggle/input/datasets/youssefashraf57/real-da...,Aggressive,5,1.666625,2.257420,0.960,1
6,/kaggle/input/datasets/youssefashraf57/real-da...,Aggressive,6,1.967030,2.557823,0.955,1
7,/kaggle/input/datasets/youssefashraf57/real-da...,Aggressive,7,2.267433,2.858228,0.940,1
8,/kaggle/input/datasets/youssefashraf57/real-da...,Aggressive,8,2.567837,3.158633,0.950,1
9,/kaggle/input/datasets/youssefashraf57/real-da...,Aggressive,9,2.868242,3.459038,0.975,1



================ SUMMARY BY SESSION TYPE ================



,session_type,sessions,total_windows,predicted_crash_windows,avg_crash_ratio,avg_max_probability,avg_mean_probability
0,Aggressive,2,1249.0,1249.0,1.0,1.000000,0.930440
1,Bump,9,1755.0,1755.0,1.0,0.990556,0.942382
2,Pothole,8,1623.0,1623.0,1.0,0.995000,0.964715
3,Slow,2,3373.0,3373.0,1.0,1.000000,0.965441
4,Standard,2,1217.0,1217.0,1.0,1.000000,0.924987



================ ONLY SESSIONS WITH PREDICTED CRASH WINDOWS ================



,session,session_type,num_rows,num_windows,predicted_crash_windows,crash_window_ratio,max_crash_probability,mean_crash_probability,status
0,/kaggle/input/datasets/youssefashraf57/real-da...,Slow,52101.0,1735.0,1735.0,1.0,1.000,0.977928,ok
1,/kaggle/input/datasets/youssefashraf57/real-da...,Slow,49197.0,1638.0,1638.0,1.0,1.000,0.952953,ok
2,/kaggle/input/datasets/youssefashraf57/real-da...,Aggressive,37501.0,1249.0,1249.0,1.0,1.000,0.930440,ok
3,/kaggle/input/datasets/youssefashraf57/real-da...,Bump,36133.0,1203.0,1203.0,1.0,1.000,0.976101,ok
4,/kaggle/input/datasets/youssefashraf57/real-da...,Pothole,31901.0,1062.0,1062.0,1.0,1.000,0.948559,ok
5,/kaggle/input/datasets/youssefashraf57/real-da...,Standard,20601.0,685.0,685.0,1.0,1.000,0.923916,ok
6,/kaggle/input/datasets/youssefashraf57/real-da...,Standard,16001.0,532.0,532.0,1.0,1.000,0.926057,ok
7,/kaggle/input/datasets/youssefashraf57/real-da...,Bump,10001.0,332.0,332.0,1.0,1.000,0.963901,ok
8,/kaggle/input/datasets/youssefashraf57/real-da...,Pothole,6701.0,222.0,222.0,1.0,1.000,0.971644,ok
9,/kaggle/input/datasets/youssefashraf57/real-da...,Pothole,3601.0,119.0,119.0,1.0,0.995,0.972773,ok


In [ ]:
# =========================================================
# TEST ALL REAL SESSIONS USING CNN + GRU MODEL
# =========================================================

import pandas as pd
import numpy as np
import os
from glob import glob

# ---------------------------------------------------------
# PATH
# ---------------------------------------------------------
base_path = "/kaggle/input/datasets/youssefashraf57/real-data/Road Data"

# ---------------------------------------------------------
# FINAL MODEL
# ---------------------------------------------------------
final_model = cnn_gru_model

# ---------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------
window_size = 60
step_size = 30

feature_columns = [
    "accel_x","accel_y","accel_z",
    "gyro_x","gyro_y","gyro_z",
    "speed"
]

# ---------------------------------------------------------
# FIND ALL SESSION FOLDERS
# ---------------------------------------------------------
all_accel_files = glob(os.path.join(base_path,"**","Accelerometer.csv"),recursive=True)

session_folders=[]

for f in all_accel_files:

    folder=os.path.dirname(f)

    if os.path.exists(os.path.join(folder,"Gyroscope.csv")) and \
       os.path.exists(os.path.join(folder,"Location.csv")):

        session_folders.append(folder)

session_folders=sorted(list(set(session_folders)))

print("Total sessions:",len(session_folders))

# ---------------------------------------------------------
# LOAD SESSION
# ---------------------------------------------------------
def load_session(folder):

    accel=pd.read_csv(folder+"/Accelerometer.csv")
    gyro=pd.read_csv(folder+"/Gyroscope.csv")
    loc=pd.read_csv(folder+"/Location.csv")

    accel=accel.rename(columns={
        "seconds_elapsed":"timestamp",
        "x":"accel_x",
        "y":"accel_y",
        "z":"accel_z"
    })

    gyro=gyro.rename(columns={
        "seconds_elapsed":"timestamp",
        "x":"gyro_x",
        "y":"gyro_y",
        "z":"gyro_z"
    })

    loc=loc.rename(columns={
        "seconds_elapsed":"timestamp"
    })

    accel=accel[["timestamp","accel_x","accel_y","accel_z"]]
    gyro=gyro[["timestamp","gyro_x","gyro_y","gyro_z"]]
    loc=loc[["timestamp","speed"]]

    accel=accel.sort_values("timestamp")
    gyro=gyro.sort_values("timestamp")
    loc=loc.sort_values("timestamp")

    df=pd.merge_asof(accel,gyro,on="timestamp",direction="nearest")
    df=pd.merge_asof(df,loc,on="timestamp",direction="nearest")

    df=df.dropna().reset_index(drop=True)

    return df

# ---------------------------------------------------------
# WINDOW CREATION
# ---------------------------------------------------------
def create_windows(df):

    X=[]
    times=[]

    if len(df)<window_size:
        return np.array([]),[]

    for i in range(0,len(df)-window_size+1,step_size):

        w=df.iloc[i:i+window_size]

        X.append(w[feature_columns].values)
        times.append((w["timestamp"].iloc[0],w["timestamp"].iloc[-1]))

    return np.array(X),times

# ---------------------------------------------------------
# SESSION TYPE
# ---------------------------------------------------------
def session_type(path):

    p=path.lower()

    if "aggressive" in p:
        return "Aggressive"

    if "standard" in p:
        return "Standard"

    if "slow" in p:
        return "Slow"

    if "bump" in p:
        return "Bump"

    if "pothole" in p:
        return "Pothole"

    return "Unknown"

# ---------------------------------------------------------
# RUN ALL SESSIONS
# ---------------------------------------------------------
session_results=[]
window_results=[]

for folder in session_folders:

    print("Processing:",folder)

    try:

        df=load_session(folder)

        X,times=create_windows(df)

        if len(X)==0:
            continue

        # SCALE USING TRAINING SCALER
        X2=X.reshape(-1,len(feature_columns))
        X_scaled=scaler.transform(X2).reshape(X.shape)

        # CNN/GRU expects 3D input
        y_prob=final_model.predict(X_scaled,verbose=0).flatten()

        y_pred=(y_prob>=0.95).astype(int)

        temp=pd.DataFrame({

            "session":folder,
            "session_type":session_type(folder),
            "window":range(len(y_pred)),
            "start":[t[0] for t in times],
            "end":[t[1] for t in times],
            "prob":y_prob,
            "pred":y_pred
        })

        window_results.append(temp)

        session_results.append({

            "session":folder,
            "session_type":session_type(folder),
            "windows":len(y_pred),
            "predicted_crash":int(y_pred.sum()),
            "ratio":float(y_pred.mean()),
            "max_prob":float(y_prob.max()),
            "mean_prob":float(y_prob.mean())
        })

    except Exception as e:

        print("Error:",e)

# ---------------------------------------------------------
# RESULTS
# ---------------------------------------------------------
session_df=pd.DataFrame(session_results)
window_df=pd.concat(window_results,ignore_index=True)

print("\nSESSION SUMMARY\n")
display(session_df.sort_values("predicted_crash",ascending=False))

print("\nWINDOW SAMPLE\n")
display(window_df.head())

print("\nSUMMARY BY TYPE\n")

display(
    session_df.groupby("session_type").agg(
        sessions=("session","count"),
        total_windows=("windows","sum"),
        predicted_crash=("predicted_crash","sum"),
        avg_ratio=("ratio","mean"),
        avg_max_prob=("max_prob","mean"),
        avg_mean_prob=("mean_prob","mean")
    ).reset_index()
)

# # ---------------------------------------------------------
# # SAVE RESULTS
# # ---------------------------------------------------------
# session_df.to_csv("/kaggle/working/cnn_gru_real_sessions.csv",index=False)
# window_df.to_csv("/kaggle/working/cnn_gru_real_windows.csv",index=False)

# print("\nSaved to /kaggle/working/")

Total sessions: 23
Processing: /kaggle/input/datasets/youssefashraf57/real-data/Road Data/Driving Behaviour/1. Aggressive
Processing: /kaggle/input/datasets/youssefashraf57/real-data/Road Data/Driving Behaviour/2. Aggressive
Error: "None of [Index(['timestamp', 'accel_x', 'accel_y', 'accel_z'], dtype='object')] are in the [columns]"
Processing: /kaggle/input/datasets/youssefashraf57/real-data/Road Data/Driving Behaviour/3. Standard
Processing: /kaggle/input/datasets/youssefashraf57/real-data/Road Data/Driving Behaviour/4. Standard
Processing: /kaggle/input/datasets/youssefashraf57/real-data/Road Data/Driving Behaviour/5. Slow
Processing: /kaggle/input/datasets/youssefashraf57/real-data/Road Data/Driving Behaviour/6. Slow
Processing: /kaggle/input/datasets/youssefashraf57/real-data/Road Data/Road Anomalies/1. Bump
Processing: /kaggle/input/datasets/youssefashraf57/real-data/Road Data/Road Anomalies/10. Pothole
Processing: /kaggle/input/datasets/youssefashraf57/real-data/Road Data/Road A

,session,session_type,windows,predicted_crash,ratio,max_prob,mean_prob
10,/kaggle/input/datasets/youssefashraf57/real-da...,Pothole,1062,408,0.384181,0.998219,0.903504
4,/kaggle/input/datasets/youssefashraf57/real-da...,Slow,1638,29,0.017705,0.997541,0.742897
3,/kaggle/input/datasets/youssefashraf57/real-da...,Slow,1735,27,0.015562,0.997384,0.796647
0,/kaggle/input/datasets/youssefashraf57/real-da...,Aggressive,1249,12,0.009608,0.995595,0.663717
20,/kaggle/input/datasets/youssefashraf57/real-da...,Bump,332,8,0.024096,0.990945,0.782815
13,/kaggle/input/datasets/youssefashraf57/real-da...,Pothole,72,5,0.069444,0.970477,0.793053
21,/kaggle/input/datasets/youssefashraf57/real-da...,Bump,1203,3,0.002494,0.992770,0.693589
1,/kaggle/input/datasets/youssefashraf57/real-da...,Standard,685,2,0.002920,0.959345,0.681045
11,/kaggle/input/datasets/youssefashraf57/real-da...,Pothole,119,1,0.008403,0.959071,0.720891
2,/kaggle/input/datasets/youssefashraf57/real-da...,Standard,532,1,0.001880,0.984763,0.611383



WINDOW SAMPLE



,session,session_type,window,start,end,prob,pred
0,/kaggle/input/datasets/youssefashraf57/real-da...,Aggressive,0,0.164552,0.755402,0.749519,0
1,/kaggle/input/datasets/youssefashraf57/real-da...,Aggressive,1,0.465013,1.055805,0.696722,0
2,/kaggle/input/datasets/youssefashraf57/real-da...,Aggressive,2,0.765415,1.356208,0.618859,0
3,/kaggle/input/datasets/youssefashraf57/real-da...,Aggressive,3,1.065819,1.656612,0.727588,0
4,/kaggle/input/datasets/youssefashraf57/real-da...,Aggressive,4,1.366222,1.957016,0.674864,0



SUMMARY BY TYPE



,session_type,sessions,total_windows,predicted_crash,avg_ratio,avg_max_prob,avg_mean_prob
0,Aggressive,1,1249,12,0.009608,0.995595,0.663717
1,Bump,9,1755,11,0.002954,0.937518,0.751647
2,Pothole,8,1623,414,0.057754,0.949178,0.797211
3,Slow,2,3373,56,0.016633,0.997463,0.769772
4,Standard,2,1217,3,0.002400,0.972054,0.646214
